In [ ]:
# for linux
# !apt-get install poppler-utils tesseract-ocr libmagic-dev

# for mac
# !brew install poppler tesseract libmagic

In [6]:
%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain_chroma 
%pip install pymupdf
%pip install -Uq langchain langchain-community langchain-openai 
%pip install -Uq python_dotenv
%pip install langchain-groq
%pip install ipywidgets


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note

In [7]:
import json
from typing import List

# LangChain components

from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import fitz
import base64
import json
import os
# file_path = "./docs/2_Sikafloor-263 SL.pdf"
file_path = "./docs/expanded-metal-catalogue.pdf"


def page_to_base64(pdf_path: str, page_num: int, dpi: int = 150) -> str:
    """Convert a PDF page to base64 JPEG image"""
    doc = fitz.open(pdf_path)
    page = doc[page_num - 1]
    pix = page.get_pixmap(dpi=dpi)
    img_b64 = base64.b64encode(pix.tobytes('jpeg')).decode()
    doc.close()
    return img_b64


def extract_page_catalogue(pdf_path, page_num, llm, context_products=None) -> dict:
    """Ask GPT-4o to extract product catalogue data from a single page"""
    img_b64 = page_to_base64(pdf_path, page_num)
    context_hint = ""
    context_hint = ""
    if context_products:
        products_str = ', '.join(f'"{p}"' for p in context_products)
        context_hint = f"""

    Note: The previous page introduced product(s): {products_str}.
    If any heading on THIS page looks like a SECTION HEADER rather than a product name
    (e.g. "PRODUCT INFORMATION", "TECHNICAL INFORMATION", "APPLICATION INFORMATION",
    "SYSTEM INFORMATION", "BASIS OF PRODUCT DATA", "MAINTENANCE", "ECOLOGY", etc.),
    do NOT use it as product_name — set product_name to null instead.
    The calling code will automatically assign items to the correct product.
    """
    prompt = """This is a page from a building materials product catalogue.

Extract ALL product data from this page and return a JSON ARRAY. Each element represents one product section:
[
  {
    \"product_name\": \"ARCH LINTELS\",
    \"description\": \"brief product description\",
    \"items\": [
      {\"item_number\": \"SMAB_F_00054403\", \"attribute1\": \"value1\", \"attribute2\": \"value2\"},
      ...
    ]
  },
  {
    \"product_name\": \"LINTELS BRACKETS\",
    \"description\": \"...\",
    \"items\": [...]
  }
]

Rules:
- If the page is a cover/title page with a prominent product or brand name but no table, return: [{"product_name": "THE TITLE", "description": "...", "items": []}]
- Only return [] if the page has absolutely no useful information (blank, pure legal text, etc.)
- Use the exact column names from each table as attribute keys
- product_name MUST be a large heading/title visibly printed on THIS page. Do NOT guess or infer from content.
- If there is no clear large title, set product_name to null
- A product entry with only a title and no table items is valid: {\"product_name\": \"TITLE\", \"description\": \"...\", \"items\": []}
- If there is table data but no clear title, extract items with product_name set to null
- Only return [] if the page truly has NO table data at all (cover page, index page, etc.)
- Return ONLY a valid JSON array, no extra text""" + context_hint

    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
    ])
    
    response = llm.invoke([msg]).content.strip()
    if response.startswith('```'):
        response = response.split('```')[1]
        if response.startswith('json'):
            response = response[4:]
    try:
        parsed = json.loads(response)
        return parsed if isinstance(parsed, list) else [parsed]
    except json.JSONDecodeError:
        return []

from langchain_groq import ChatGroq

def build_catalogue(pdf_path: str, start_page: int = 1, end_page: int = None) -> dict:
    """逐页用 GPT-4o 提取产品目录。当某页没有标题时，沿用上一页的类目"""
    # llm = ChatOpenAI(model='gpt-4o', temperature=0)
    llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    doc.close()

    end_page = end_page or total_pages
    catalogue = {}
    last_product_name = None   # 最近一次成功归类的类目
    prev_product_names = []    # 前一页 GPT 返回的所有标题（未经回退）
    print(f'Processing pages {start_page} to {end_page} of {total_pages}...')

    for page_num in range(start_page, end_page + 1):
        print(f'  Page {page_num}/{total_pages}...', end=' ')
        results = extract_page_catalogue(pdf_path, page_num, llm, context_products=prev_product_names if prev_product_names else None)
                                

        page_had_data = False
        cur_product_names = []  # 本页 GPT 返回的所有标题

        for entry in results:
            product_name = entry.get('product_name')
            items = entry.get('items', [])

            if product_name:
                cur_product_names.append(product_name)

            # 本条目没有标题但有数据 → 找回退类目
            if not product_name and items:
                if prev_product_names:
                    product_name = prev_product_names[-1]
                    print(f'\u21a9\ufe0f  用前一页标题: {product_name}', end=' ')
                elif last_product_name:
                    product_name = last_product_name
                    print(f'\u21a9\ufe0f  沿用上一类目: {product_name}', end=' ')

            if product_name and items:
                if product_name not in catalogue:
                    catalogue[product_name] = {'description': entry.get('description', ''), 'items': []}
                existing_ids = {item.get('item_number') for item in catalogue[product_name]['items']}
                new_items = [item for item in items if item.get('item_number') not in existing_ids]
                catalogue[product_name]['items'].extend(new_items)
                last_product_name = product_name
                print(f'\u2705 {product_name} ({len(new_items)} items)', end=' ')
                page_had_data = True
            elif product_name and not items:
                if product_name not in catalogue:
                    catalogue[product_name] = {'description': entry.get('description', ''), 'items': []}
                # title-only 页：记录标题，等下一页数据来
                print(f'\U0001f4cc {product_name} (title only)', end=' ')
                page_had_data = True

        if not page_had_data:
            print('skipped', end='')
        print()

        prev_product_names = cur_product_names  # 更新前一页标题列表

        # 每页完成后写入 JSON，前端刷新即可看到最新数据
        with open('gpt_catalogue_progress.json', 'w', encoding='utf-8') as f:
            json.dump(catalogue, f, ensure_ascii=False)

    print(f'\n\u2705 Done! Found {len(catalogue)} products')
    return catalogue

gpt_catalogue = build_catalogue(file_path, start_page=11, end_page=12)

for product_name, data in gpt_catalogue.items():
    print(f'\n=== {product_name} ({len(data["items"])} items) ===')
    print(f'  Description: {data["description"]}')
    for item in data['items']:
        print(f'  {item}')

Processing pages 11 to 12 of 39...
  Page 11/39... 📌 ADJUSTABLE WALL TIE (AWT) (title only) 
  Page 12/39... ↩️  用前一页标题: ADJUSTABLE WALL TIE (AWT) ✅ ADJUSTABLE WALL TIE (AWT) (12 items) 

✅ Done! Found 1 products

=== ADJUSTABLE WALL TIE (AWT) (12 items) ===
  Description: Device for connecting a masonry leaf across a cavity to another masonry leaf or to a structural frame to resist tension and compression forces while allowing limited differential movement in the plane of the wall.
  {'Tie': 'B/P/30', 'Reference Code': 'B/P/30', 'Description': 'Brick Facing Tie', 'Section Size (mm)': '20 x 2.0 25 x 2.0', 'Length 0 - 20 (mm)': '75', 'Length 15 - 45 (mm)': '100', 'Length 40 -70 (mm)': '150', 'Length 65 - 70 (mm)': '200', 'Position of Drip (min) mm': None}
  {'Tie': 'B/D/30', 'Reference Code': 'B/D/30', 'Description': 'Brick Facing Tie', 'Section Size (mm)': '20 x 2.0 25 x 2.0', 'Length 0 - 20 (mm)': None, 'Length 15 - 45 (mm)': '100', 'Length 40 -70 (mm)': '150', 'Length 65 - 70 (mm)': 

In [15]:
import json, os, threading, webbrowser, io, warnings
from http.server import HTTPServer, SimpleHTTPRequestHandler

# 1. Export catalogue to JSON
with open('catalogue_data.json', 'w', encoding='utf-8') as f:
    json.dump(gpt_catalogue, f, indent=2, ensure_ascii=False)
with open('gpt_catalogue_progress.json', 'w', encoding='utf-8') as f:
    json.dump(gpt_catalogue, f, indent=2, ensure_ascii=False)
print('✅ Exported catalogue_data.json')

# 2. Shared upload status
upload_status = {'state': 'idle', 'message': '', 'progress': 0, 'catalogue': {}}

class CatalogueHandler(SimpleHTTPRequestHandler):
    def log_message(self, format, *args): pass

    def do_OPTIONS(self):
        self.send_response(200)
        self._cors()
        self.end_headers()

    def do_GET(self):
        if self.path == '/upload-status':
            self._send_json(upload_status)
        else:
            super().do_GET()

    def do_POST(self):
        try:
            if self.path != '/upload':
                self._send_json({'error': 'Not found'}, 404)
                return

            content_type = self.headers.get('Content-Type', '')
            length = int(self.headers.get('Content-Length', 0))

            # Parse multipart with cgi.FieldStorage (most reliable in Python 3.12)
            import cgi
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', DeprecationWarning)
                form = cgi.FieldStorage(
                    fp=io.BytesIO(self.rfile.read(length)),
                    headers=self.headers,
                    environ={
                        'REQUEST_METHOD': 'POST',
                        'CONTENT_TYPE': content_type,
                        'CONTENT_LENGTH': str(length),
                    }
                )

            if 'pdf' not in form:
                self._send_json({'error': 'No PDF file received'}, 400)
                return

            pdf_item = form['pdf']
            pdf_data = pdf_item.file.read()
            pdf_name = pdf_item.filename or 'upload.pdf'

            try: start_page = int(form.getvalue('start_page', '1'))
            except: start_page = 1
            end_page_val = form.getvalue('end_page', '')
            try: end_page = int(end_page_val) if end_page_val else None
            except: end_page = None

            # Save uploaded PDF
            upload_dir = os.path.join(os.path.dirname(os.path.abspath('frontend.html')), 'uploads')
            os.makedirs(upload_dir, exist_ok=True)
            pdf_path = os.path.join(upload_dir, pdf_name)
            with open(pdf_path, 'wb') as f:
                f.write(pdf_data)

            upload_status.update({'state': 'processing', 'message': f'Processing {pdf_name}…', 'progress': 5, 'catalogue': {}})

            def process():
                try:
                    result = build_catalogue(pdf_path, start_page=start_page, end_page=end_page)
                    with open('catalogue_data.json', 'w', encoding='utf-8') as f:
                        json.dump(result, f, indent=2, ensure_ascii=False)
                    with open('gpt_catalogue_progress.json', 'w', encoding='utf-8') as f:
                        json.dump(result, f, indent=2, ensure_ascii=False)
                    upload_status.update({'state': 'done', 'message': f'Done! Found {len(result)} products.', 'progress': 100, 'catalogue': result})
                except Exception as e:
                    upload_status.update({'state': 'error', 'message': str(e), 'progress': 0})

            threading.Thread(target=process, daemon=True).start()
            self._send_json({'status': 'processing'})

        except Exception as e:
            self._send_json({'error': f'Server error: {str(e)}'}, 500)

    def _cors(self):
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')

    def _send_json(self, data, code=200):
        body = json.dumps(data).encode()
        self.send_response(code)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', len(body))
        self._cors()
        self.end_headers()
        self.wfile.write(body)

PORT = 8765

# Kill any process already using the port (macOS/Linux)
import subprocess, signal
result = subprocess.run(['lsof', '-ti', f':{PORT}'], capture_output=True, text=True)
for pid in result.stdout.strip().split():
    try:
        os.kill(int(pid), signal.SIGTERM)
        print(f'🔪 Killed old server (pid {pid})')
    except: pass

import time; time.sleep(0.5)  # wait for port to free

def start_server():
    os.chdir(os.path.dirname(os.path.abspath('frontend.html')))
    server = HTTPServer(('localhost', PORT), CatalogueHandler)
    server.serve_forever()

t = threading.Thread(target=start_server, daemon=True)
t.start()
print(f'✅ Server started on port {PORT} (with upload support)')

url = f'http://localhost:{PORT}/frontend.html'
webbrowser.open(url)
print(f'🌐 Opening {url}')

: 